# 🐱 MEL WAN LoRA Training
訓練 WAN 2.2 專用 MEL 角色 LoRA

**硬體需求**: T4 16GB+ (blocks_to_swap offload)
**訓練時間**: ~2小時 (28圖, 4 epochs)
**輸出**: MEL WAN LoRA (.safetensors)

In [ ]:
# @title 1. 環境設置
import os, sys, json
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Clone lora_trainer
%cd /content
!git clone https://github.com/your-repo/lora_trainer.git 2>/dev/null || echo "Repo exists or manual upload"

# Install deps
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q accelerate transformers sentencepiece protobuf einops huggingface_hub safetensors peft deepspeed
!pip install -q bitsandbytes flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn optional"

# Copy training images from Drive (if uploaded there)
# Or upload dataset directly via Colab file browser
DATASET_DIR = Path('/content/drive/MyDrive/mel_wan_dataset')
if not DATASET_DIR.exists():
    print("⚠️ Upload dataset to Google Drive: MyDrive/mel_wan_dataset/")
    print("   Expected structure:")
    print("   mel_wan_dataset/")
    print("   ├── images/  (28 files)")
    print("   └── captions/ (28 .txt files)")

print("✅ Setup done")

In [ ]:
# @title 2. 下載 WAN 2.2 模型
from huggingface_hub import snapshot_download
import torch

MODEL_DIR = Path('/content/wan22_model')
QWEN3_DIR = Path('/content/qwen3_06b')

# Download WAN 2.2 I2V DiT (choose one):
print("Downloading WAN 2.2 DiT (~28GB, may take 10-20 min)...")
# Option A: Full safetensors from Wan-AI
!huggingface-cli download Wan-AI/Wan2.2-I2V-A14B --local-dir {MODEL_DIR} \
  --include "diffusion_pytorch_model*.safetensors" 2>&1 | tail -5

# Option B: Single-file safetensors (if available)
# model_path = snapshot_download("Wan-AI/Wan2.2-I2V-A14B", local_dir=MODEL_DIR)

# Download Qwen3 0.6B text encoder
print("Downloading Qwen3 0.6B...")
!huggingface-cli download Qwen/Qwen3-0.6B --local-dir {QWEN3_DIR} 2>&1 | tail -3

print("✅ Models ready")
!ls -lh {MODEL_DIR}/ | head -10

In [ ]:
# @title 3. 準備訓練資料集
import json
from pathlib import Path

IMG_DIR = Path('/content/drive/MyDrive/mel_wan_dataset/images')
CAP_DIR = Path('/content/drive/MyDrive/mel_wan_dataset/captions')

# Create metadata.jsonl
metadata = []
for img_path in sorted(IMG_DIR.glob('*')):
    if img_path.suffix.lower() not in ['.png', '.jpg', '.jpeg']:
        continue
    # Find matching caption
    cap_name = img_path.stem + '.txt'
    cap_path = CAP_DIR / cap_name
    if not cap_path.exists():
        # Try removing number prefix
        for c in CAP_DIR.glob('*.txt'):
            if img_path.stem.split('_', 1)[-1] in c.stem:
                cap_path = c
                break
    
    caption = cap_path.read_text().strip() if cap_path.exists() else "mel_character"
    metadata.append({
        "image": str(img_path),
        "caption": caption
    })

print(f"Dataset: {len(metadata)} images")
for m in metadata[:3]:
    print(f"  {Path(m['image']).name}: {m['caption'][:60]}...")

# Save metadata
with open('/content/dataset.json', 'w') as f:
    for m in metadata:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')
print("✅ dataset.json saved")

# Also create a TOML config for anima trainer
toml_config = f'''
pretrained_model_name_or_path = "/content/wan22_model/diffusion_pytorch_model.safetensors"
qwen3 = "/content/qwen3_06b"
vae = "/content/wan22_model/vae"

output_dir = "/content/drive/MyDrive/mel_wan_lora_output"
output_name = "mel_wan_lora_v1"

dataset_config = "/content/dataset.json"

resolution = "832,480"
batch_size = 1
max_train_epochs = 4
save_every_n_epochs = 1

network_dim = 8
network_alpha = 16

learning_rate = 1e-4
lr_scheduler = "cosine"
lr_warmup_steps = 50
optimizer_type = "adamw8bit"

mixed_precision = "bf16"
gradient_checkpointing = true
blocks_to_swap = 28
cpu_offload_checkpointing = true

cache_latents = true
cache_text_encoder_outputs = true

attn_mode = "torch"
split_attn = false

discrete_flow_shift = 5.0
weighting_scheme = "none"
logit_mean = 0.0
logit_std = 1.0
mode_scale = 1.0

log_with = "tensorboard"
logging_dir = "/content/logs"

sample_prompts = "/content/sample_prompts.txt"
'''
with open('/content/train_config.toml', 'w') as f:
    f.write(toml_config)

# Sample prompts
with open('/content/sample_prompts.txt', 'w') as f:
    f.write('{"prompt": "mel, silver hair, purple eyes, cat ears, school uniform, rooftop sunset, dancing", "negative_prompt": "worst quality, blurry"}')

print("✅ train_config.toml ready")

In [ ]:
# @title 4. 開始訓練 🔥
import sys, os
sys.path.insert(0, '/content/lora_trainer')

!cd /content/lora_trainer && python anima_train_network.py \
  --config_file /content/train_config.toml \
  2>&1 | tee /content/train_log.txt

print("\n✅ Training complete! Check /content/drive/MyDrive/mel_wan_lora_output/")

## 📝 注意事項
- 首次運行需上傳資料集到 Google Drive: `MyDrive/mel_wan_dataset/`
- 模型下載約需 30GB 空間
- T4 16GB 使用 blocks_to_swap=28 + CPU offload 可訓練
- LoRA 輸出會自動存到 Google Drive